<a href="https://colab.research.google.com/github/dhiyanesh-m-cse/daaEX/blob/main/daaex02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import string

def naive_search(text, pattern):
  n, m = len(text), len(pattern)
  matches, comparisons = [], 0
  for i in range(n - m + 1):
    j = 0
    while j < m:
      comparisons += 1
      if text[i + j] != pattern[j]:
        break
      j += 1
    if j == m:
      matches.append(i)
  return matches, comparisons

def compute_lps(pattern):
  m = len(pattern)
  lps = [0] * m
  length, i = 0, 1
  while i < m:
    if pattern[i] == pattern[length]:
      length += 1
      lps[i] = length
      i += 1
    elif length != 0:
      length = lps[length - 1]
    else:
      lps[i] = 0
      i += 1
  return lps

def kmp_search(text, pattern):
  n, m = len(text), len(pattern)
  lps = compute_lps(pattern)
  matches, comparisons = [], 0
  i = j = 0
  while i < n:
    comparisons += 1
    if pattern[j] == text[i]:
      i += 1; j += 1
    if j == m:
      matches.append(i - j)
      j = lps[j - 1]
    elif i < n and pattern[j] != text[i]:
      if j != 0:
        j = lps[j - 1]
      else:
        i += 1
  return matches, comparisons

def rabin_karp(text, pattern, q=101):
  n, m = len(text), len(pattern)
  d = 256
  h = pow(d, m - 1, q)
  p_hash = t_hash = 0
  matches, comparisons = [], 0

  for i in range(m):
    p_hash = (d * p_hash + ord(pattern[i])) % q
    t_hash = (d * t_hash + ord(text[i])) % q

  for s in range(n - m + 1):
    if p_hash == t_hash:
      # Check characters one by one
      match = True
      for k in range(m):
        comparisons += 1
        if text[s + k] != pattern[k]:
          match = False
          break
      if match:
        matches.append(s)

    # Update hash for next window
    if s < n - m:
      t_hash = (d * (t_hash - ord(text[s]) * h) + ord(text[s + m])) % q
      if t_hash < 0:
        t_hash += q
  return matches, comparisons

# --- Main Execution ---
text = 'AABAACAADAABAABA'
pattern = 'AABA'
print(f'Text:    {text}')
print(f'Pattern: {pattern}')

m1, c1 = naive_search(text, pattern)
m2, c2 = kmp_search(text, pattern)
m3, c3 = rabin_karp(text, pattern)
print(f'\nNaive  -> Matches at: {m1}, Comparisons: {c1}')
print(f'KMP    -> Matches at: {m2}, Comparisons: {c2}')
print(f'RK     -> Matches at: {m3}, Comparisons: {c3}')

# Performance comparison
text_large = ''.join(random.choices('ABCD', k=10000))
patterns = ['AB', 'ABCD', 'ABCDAB', 'ABCDABCD']

print(f'\n{"Pattern":>12} {"Naive":>10} {"KMP":>10} {"RK":>10}')
print('-' * 50)
for p in patterns:
    _, c1 = naive_search(text_large, p)
    _, c2 = kmp_search(text_large, p)
    _, c3 = rabin_karp(text_large, p)
    print(f'{p:>12} {c1:>10} {c2:>10} {c3:>10}')

Text:    AABAACAADAABAABA
Pattern: AABA

Naive  -> Matches at: [0, 9, 12], Comparisons: 30
KMP    -> Matches at: [0, 9, 12], Comparisons: 18
RK     -> Matches at: [0, 9, 12], Comparisons: 12

     Pattern      Naive        KMP         RK
--------------------------------------------------
          AB      12500      10000       1270
        ABCD      13308      10000        262
      ABCDAB      13366      10014        139
    ABCDABCD      13365      10014        152


## Minimum Spanning Tree (MST) Algorithms: Kruskal's and Prim's

This section demonstrates two classic algorithms for finding the Minimum Spanning Tree (MST) of a connected, undirected graph with weighted edges:

1.  **Kruskal's Algorithm**
2.  **Prim's Algorithm**

Both algorithms aim to find a subset of the edges that forms a tree containing every vertex, such that the sum of the weights of the edges in the tree is minimized.

In [ ]:
import heapq

# --- Union-Find for Kruskal ---
class UnionFind:
  def __init__(self, n):
    self.parent = list(range(n))
    self.rank   = [0] * n

  def find(self, x):
    if self.parent[x] != x:
      self.parent[x] = self.find(self.parent[x])  # Path compression
    return self.parent[x]

  def union(self, x, y):
    rx, ry = self.find(x), self.find(y)
    if rx == ry: return False
    if self.rank[rx] < self.rank[ry]: rx, ry = ry, rx
    self.parent[ry] = rx
    if self.rank[rx] == self.rank[ry]: self.rank[rx] += 1
    return True

def kruskal(n, edges):
  """edges: list of (weight, u, v)"""
  edges.sort()  # O(E log E)
  uf   = UnionFind(n)
  mst  = []
  cost = 0
  for w, u, v in edges:
    if uf.union(u, v):
      mst.append((u, v, w))
      cost += w
      if len(mst) == n - 1:
        break
  return mst, cost

def prim(n, adj, start=0):
  """adj: adjacency list {u: [(v, w), ...]}"""
  INF    = float('inf')
  key    = [INF] * n
  parent = [-1]  * n
  inMST  = [False] * n

  key[start] = 0
  pq = [(0, start)]

  mst = []
  cost = 0

  while pq:
    w, u = heapq.heappop(pq)

    if inMST[u]: continue
    inMST[u] = True

    if parent[u] != -1:
      mst.append((parent[u], u, w))
      cost += w

    for v, wt in adj.get(u, []):
      if not inMST[v] and wt < key[v]:
        key[v] = wt
        parent[v] = u
        heapq.heappush(pq, (wt, v))

  return mst, cost

### Graph Definition

The graph is defined with `n = 7` vertices (numbered 0 to 6) and a list of `edges`. Each edge is represented as a tuple `(weight, u, v)`, indicating the weight and the two connected vertices.

For Prim's algorithm, an adjacency list `adj` is constructed from the `edges` list, where each key `u` maps to a list of tuples `(v, w)` representing neighbors `v` and the weight `w` of the edge connecting `u` and `v`.

In [ ]:
import heapq

# --- Union-Find for Kruskal ---
class UnionFind:
  def __init__(self, n):
    self.parent = list(range(n))
    self.rank   = [0] * n

  def find(self, x):
    if self.parent[x] != x:
      self.parent[x] = self.find(self.parent[x])  # Path compression
    return self.parent[x]

  def union(self, x, y):
    rx, ry = self.find(x), self.find(y)
    if rx == ry: return False
    if self.rank[rx] < self.rank[ry]: rx, ry = ry, rx
    self.parent[ry] = rx
    if self.rank[rx] == self.rank[ry]: self.rank[rx] += 1
    return True

def kruskal(n, edges):
  """edges: list of (weight, u, v)"""
  edges.sort()  # O(E log E)
  uf   = UnionFind(n)
  mst  = []
  cost = 0
  for w, u, v in edges:
    if uf.union(u, v):
      mst.append((u, v, w))
      cost += w
      if len(mst) == n - 1:
        break
  return mst, cost

def prim(n, adj, start=0):
  """adj: adjacency list {u: [(v, w), ...]}"""
  INF    = float('inf')
  key    = [INF] * n
  parent = [-1]  * n
  inMST  = [False] * n

  key[start] = 0
  pq = [(0, start)]

  mst = []
  cost = 0

  while pq:
    w, u = heapq.heappop(pq)

    if inMST[u]: continue
    inMST[u] = True

    if parent[u] != -1:
      mst.append((parent[u], u, w))
      cost += w

    for v, wt in adj.get(u, []):
      if not inMST[v] and wt < key[v]:
        key[v] = wt
        parent[v] = u
        heapq.heappush(pq, (wt, v))

  return mst, cost

# --- Graph Definition ---
n = 7
edges = [
    (7, 0, 1), (5, 0, 3), (8, 1, 2), (9, 1, 3),
    (7, 1, 4), (5, 2, 4), (15, 3, 4), (6, 3, 5),
    (8, 4, 5), (9, 4, 6), (11, 5, 6)
]

adj = {}
for w, u, v in edges:
    adj.setdefault(u, []).append((v, w))
    adj.setdefault(v, []).append((u, w))

k_mst, k_cost = kruskal(n, edges[:])
p_mst, p_cost = prim(n, adj)

print('=== Kruskal\'s MST ===')
for u, v, w in k_mst:
    print(f'  Edge ({u} - {v})  Weight: {w}')
print(f'  Total MST Cost: {k_cost}')

print('\n=== Prim\'s MST ===')
for u, v, w in p_mst:
    print(f'  Edge ({u} - {v})  Weight: {w}')
print(f'  Total MST Cost: {p_cost}')

=== Kruskal's MST ===
  Edge (0 - 3)  Weight: 5
  Edge (2 - 4)  Weight: 5
  Edge (3 - 5)  Weight: 6
  Edge (0 - 1)  Weight: 7
  Edge (1 - 4)  Weight: 7
  Edge (4 - 6)  Weight: 9
  Total MST Cost: 39

=== Prim's MST ===
  Edge (0 - 3)  Weight: 5
  Edge (3 - 5)  Weight: 6
  Edge (0 - 1)  Weight: 7
  Edge (1 - 4)  Weight: 7
  Edge (4 - 2)  Weight: 5
  Edge (4 - 6)  Weight: 9
  Total MST Cost: 39
